# Notebook 02: Rule-Based & Information-Theoretic Solvers
In this notebook, we implement and benchmark three classical non-neural baselines:
1. **Random Solver** (Baseline A)
2. **Letter Frequency Solver** (Baseline B)
3. **Information-Theoretic Entropy Solver** (Baseline C)

### Mathematical Formulation of Information Gain (Entropy):
For a candidate guess $g$ and remaining candidate secrets $\mathcal{C}$:
$$H(F|g) = -\sum_{f} p(f|g) \log_2 p(f|g)$$
where $p(f|g) = \frac{\text{count}(f)}{|\mathcal{C}|}$.

In [ ]:
import sys
import os
import zipfile

# Auto-extract zip file if Kaggle uploaded dataset as a raw zip
if os.path.exists("/kaggle/input"):
    for root_dir, dirs, files in os.walk("/kaggle/input"):
        for file in files:
            if file.endswith(".zip") and "wordle" in file.lower():
                zip_path = os.path.join(root_dir, file)
                print(f"Extracting uploaded Kaggle dataset: {zip_path} -> /kaggle/working/")
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall("/kaggle/working/")

# Dynamic path resolution for local and Kaggle environments
possible_roots = [
    ".",
    "..",
    "/kaggle/working",
    "/kaggle/working/wordle",
    "/kaggle/working/wordle-ai",
    "/kaggle/input/wordle",
    "/kaggle/input/wordle/wordle",
    "/kaggle/input/wordle/wordle-ai",
    "/kaggle/input/wordle-ai",
]

# Search recursively inside /kaggle/input for any folder containing 'src'
if os.path.exists("/kaggle/input"):
    for root_dir, dirs, files in os.walk("/kaggle/input"):
        if "src" in dirs:
            possible_roots.insert(0, root_dir)

for root in possible_roots:
    if os.path.exists(os.path.join(root, "src")):
        abs_root = os.path.abspath(root)
        if abs_root not in sys.path:
            sys.path.insert(0, abs_root)
        print(f"Found 'src' package at: {abs_root}")
        break

if os.path.exists("/kaggle/input"):
    print("Running on Kaggle GPU/CPU Environment!")
else:
    print("Running on Local Environment!")

from src.wordle.dictionary import WordleDictionary
from src.solvers.random_solver import RandomSolver
from src.solvers.frequency_solver import FrequencySolver
from src.solvers.entropy_solver import EntropySolver
from src.evaluation.benchmark import BenchmarkRunner

dict_mgr = WordleDictionary()
runner = BenchmarkRunner(dict_mgr)
test_set = dict_mgr.test_secrets[:30]

print("Evaluating Random Solver...")
m_random = runner.evaluate_solver(RandomSolver(dict_mgr, seed=42), test_secrets=test_set)
print(f"Random Solver Win Rate: {m_random.win_rate:.1f}%, Avg Guesses: {m_random.avg_guesses:.2f}")

print("Evaluating Frequency Solver...")
m_freq = runner.evaluate_solver(FrequencySolver(dict_mgr), test_secrets=test_set)
print(f"Frequency Solver Win Rate: {m_freq.win_rate:.1f}%, Avg Guesses: {m_freq.avg_guesses:.2f}")

print("Evaluating Entropy Solver...")
m_entropy = runner.evaluate_solver(EntropySolver(dict_mgr), test_secrets=test_set)
print(f"Entropy Solver Win Rate: {m_entropy.win_rate:.1f}%, Avg Guesses: {m_entropy.avg_guesses:.2f}")